In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(5, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [3]:
import mllabs
from mllabs.processor import PolarsLoader, ExprProcessor, PandasConverter
mllabs.__version__

'0.2.2'

In [4]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [6]:
# import shutil
# shutil.rmtree('exp/exp1')
import os

In [7]:
from mllabs import Experimenter
from mllabs.collector import MetricCollector, ModelAttrCollector, StackingCollector
from mllabs import Connector
from sklearn.metrics import roc_auc_score

if os.path.exists('exp/exp1'):
    e = Experimenter.load('exp/exp1', df_train)
else:
    e = Experimenter.create(
        df_train, 'exp/exp1', sp = StratifiedShuffleSplit(n_splits=1, random_state = 123), 
        sp_v = StratifiedShuffleSplit(n_splits=1, train_size=0.9, random_state = 123), splitter_params = {'y': y}
    )

Loaded: 29 node(s), 6 group(s), 1 fold(s)


In [11]:
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression

# Configuration

In [14]:
e.add_collector(
    MetricCollector(
        'AUC', Connector(edges = {'y': [(None, y)]}), '.*'+ y +'_1', roc_auc_score, include_train = True
    )
)
e.add_collector(
    ModelAttrCollector(
        'lgb_feature_importance', Connector(processor = lgb.LGBMClassifier), 'feature_importances'
    )
)
e.add_collector(
    StackingCollector(
        'stacking', Connector(edges = {'y': [(None, y)]}),
        '.*' + y + '_1', method='mean', experimenter = e
    )
)

e.set_grp('clf', role = 'head', method = 'predict_proba', edges = {'y': [(None, y)]})
e.set_grp('lgb', parent = 'clf', processor = lgb.LGBMClassifier, params={'verbose': -1, 'early_stopping': lgb.early_stopping(100), 'eval_metric': 'AUC'})
e.set_grp('xgb', parent = 'clf', processor = xgb.XGBClassifier)
e.set_grp('cb', parent = 'clf', processor = cb.CatBoostClassifier)
e.set_grp('lr', parent = 'clf', processor = LogisticRegression)
e.set_grp('pre', role = 'stage', method = 'transform')

Collect 1/1 (100%) Node 0


{'result': 'new',
 'grp': <mllabs._pipeline.PipelineGroup at 0x7fd7ff0c5880>,
 'affected_nodes': []}

## Stage Nodes

In [15]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
e.set_node(
    'ord', grp = 'pre', processor=OrdinalEncoder, 
    edges ={'X': [(None, 'grade_subgrade')]}, params={'categories': [np.sort(df_train['grade_subgrade'].unique())]}
)

e.set_node(
    'ohe', grp = 'pre', processor=OneHotEncoder, 
    edges ={'X': [(None, X_cat)]}, params={'sparse_output': False}
)

e.set_node(
    'std', grp = 'pre', processor=StandardScaler, edges ={'X': [(None, X_num)]}
)
e.build()

Building 3 node(s)
Build 1/1 (100%) ord 3/3 (100%)
Build complete: 3 node(s)


## LightGBM

In [16]:
e.set_node('lgb1', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.1})
e.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb1 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.9056, valid_0-binary_logloss: 0.4522, valid_1-auc: 0.9070, valid_1-binary_logloss: 0.4522Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[563]	valid_0's auc: 0.936376	valid_0's binary_logloss: 0.226115	valid_1's auc: 0.923525	valid_1's binary_logloss: 0.242813
Exp 1/1 (100%) lgb1 1/1 (100%)
Experimentation complete: 1 node(s)


In [17]:
e.set_node(
    'lgb2', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05}
)
e.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb2 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.9056, valid_0-binary_logloss: 0.4764, valid_1-auc: 0.9070, valid_1-binary_logloss: 0.4764Training until validation scores don't improve for 100 rounds
Exp 0/1 (0%) > lgb2 0/1 (0%) > 1000/10000 (10%) valid_0-auc: 0.9349, valid_0-binary_logloss: 0.2283, valid_1-auc: 0.9234, valid_1-binary_logloss: 0.2429Early stopping, best iteration is:
[1356]	valid_0's auc: 0.939157	valid_0's binary_logloss: 0.222697	valid_1's auc: 0.923553	valid_1's binary_logloss: 0.242704
Exp 1/1 (100%) lgb2 1/1 (100%)
Experimentation complete: 1 node(s)


In [18]:
e.add_collector(
    ModelAttrCollector(
        'lgb_evals_results', 
        Connector(processor=lgb.LGBMClassifier),
        'evals_result'
    )
)

Collect 1/1 (100%) lgb1 2/2 (100%)


In [19]:
e.set_node(
    'lgb3', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.2}
)
e.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb3 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.9056, valid_0-binary_logloss: 0.4086, valid_1-auc: 0.9070, valid_1-binary_logloss: 0.4086Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[205]	valid_0's auc: 0.931672	valid_0's binary_logloss: 0.231746	valid_1's auc: 0.922612	valid_1's binary_logloss: 0.24429
Exp 1/1 (100%) lgb3 1/1 (100%)
Experimentation complete: 1 node(s)


In [20]:
e.set_node(
    'lgb4', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075}
)
e.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb4 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.9056, valid_0-binary_logloss: 0.4641, valid_1-auc: 0.9070, valid_1-binary_logloss: 0.4641Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[617]	valid_0's auc: 0.933892	valid_0's binary_logloss: 0.229473	valid_1's auc: 0.92303	valid_1's binary_logloss: 0.243213
Exp 1/1 (100%) lgb4 1/1 (100%)
Experimentation complete: 1 node(s)


In [21]:
e.set_node(
    'lgb5', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075, 'num_leaves': 15}
)

{'result': 'new',
 'affected_nodes': [],
 'old_obj': None,
 'obj': <mllabs._pipeline.PipelineNode at 0x7fd7c0101f70>}

In [22]:
e.set_node(
    'lgb6', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.1, 'num_leaves': 15}
)
e.exp()

Experimenting 2 node(s)
Exp 0/1 (0%) > lgb6 0/2 (0%) > 1/10000 (0%) valid_0-auc: 0.9009, valid_0-binary_logloss: 0.4531, valid_1-auc: 0.9028, valid_1-binary_logloss: 0.4531Training until validation scores don't improve for 100 rounds
Exp 0/1 (0%) > lgb6 0/2 (0%) > 1000/10000 (10%) valid_0-auc: 0.9337, valid_0-binary_logloss: 0.2295, valid_1-auc: 0.9238, valid_1-binary_logloss: 0.2424Early stopping, best iteration is:
[995]	valid_0's auc: 0.933689	valid_0's binary_logloss: 0.229569	valid_1's auc: 0.923844	valid_1's binary_logloss: 0.242419
Exp 0/1 (0%) > lgb5 1/2 (50%) > 1/10000 (0%) valid_0-auc: 0.9009, valid_0-binary_logloss: 0.4648, valid_1-auc: 0.9028, valid_1-binary_logloss: 0.4648Training until validation scores don't improve for 100 rounds
Exp 0/1 (0%) > lgb5 1/2 (50%) > 1000/10000 (10%) valid_0-auc: 0.9309, valid_0-binary_logloss: 0.2333, valid_1-auc: 0.9235, valid_1-binary_logloss: 0.2427Early stopping, best iteration is:
[1289]	valid_0's auc: 0.933446	valid_0's binary_logloss:

In [23]:
e.set_node(
    'lgb7', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05, 'num_leaves': 15}
)
e.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb7 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.9009, valid_0-binary_logloss: 0.4768, valid_1-auc: 0.9028, valid_1-binary_logloss: 0.4769Training until validation scores don't improve for 100 rounds
Exp 0/1 (0%) > lgb7 0/1 (0%) > 2000/10000 (20%) valid_0-auc: 0.9337, valid_0-binary_logloss: 0.2296, valid_1-auc: 0.9239, valid_1-binary_logloss: 0.2422Early stopping, best iteration is:
[2306]	valid_0's auc: 0.935235	valid_0's binary_logloss: 0.227504	valid_1's auc: 0.924013	valid_1's binary_logloss: 0.24214
Exp 1/1 (100%) lgb7 1/1 (100%)
Experimentation complete: 1 node(s)


In [24]:
e.set_node(
    'lgb8', grp = 'lgb', edges = {'X': [(None, X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05, 'num_leaves': 15}
)
e.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb8 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.7858, valid_0-binary_logloss: 0.4810, valid_1-auc: 0.7832, valid_1-binary_logloss: 0.4811Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[146]	valid_0's auc: 0.78967	valid_0's binary_logloss: 0.325576	valid_1's auc: 0.785937	valid_1's binary_logloss: 0.327072
Exp 1/1 (100%) lgb8 1/1 (100%)
Experimentation complete: 1 node(s)


In [25]:
e.set_node(
    'lgb9', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, 
    params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075, 'num_leaves': 63}
)
e.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb9 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.9103, valid_0-binary_logloss: 0.4636, valid_1-auc: 0.9117, valid_1-binary_logloss: 0.4637Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[403]	valid_0's auc: 0.938185	valid_0's binary_logloss: 0.223982	valid_1's auc: 0.922788	valid_1's binary_logloss: 0.243669
Exp 1/1 (100%) lgb9 1/1 (100%)
Experimentation complete: 1 node(s)


In [26]:
d = e.collectors['lgb_evals_results'].get_attrs('lgb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('valid_1', 'auc')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
    }, name = ('evals_result', 'best_iteration'))
e.pipeline.compare_nodes(
    e.pipeline.get_node_names('lgb*')
)['LGBMClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending = False)

params             \
     learning_rate num_leaves   
lgb5         0.075       15.0   
lgb6         0.100       15.0   
lgb7         0.050       15.0   
lgb2         0.050    default   
lgb1         0.100    default   
lgb4         0.075    default   
lgb9         0.075       63.0   
lgb3         0.200    default   
lgb8         0.050       15.0   

                                                                                         X  \
     DataSource [education_level, employment_status, gender, loan_purpose, marital_status]   
lgb5  [annual_income, credit_score, debt_to_income_r...                                      
lgb6  [annual_income, credit_score, debt_to_income_r...                                      
lgb7  [annual_income, credit_score, debt_to_income_r...                                      
lgb2  [annual_income, credit_score, debt_to_income_r...                                      
lgb1  [annual_income, credit_score, debt_to_income_r...                                      
lgb4  [annual_income, credit_score, debt_to_income_r...                                      
lgb9  [annual_income, credit_score, debt_to_income_r...                                      
lgb3  [annual_income, credit_score, debt_to_income_r...                                      
lgb8                                                 []                                      

           AUC                       evals_result  
         valid train_sub valid_sub best_iteration  
lgb5  0.924416  0.933446  0.923891         1288.0  
lgb6  0.924324  0.933689  0.923844          994.0  
lgb7  0.924239  0.935235  0.924013         2305.0  
lgb2  0.923872  0.939157  0.923553         1400.0  
lgb1  0.923860  0.936376  0.923525          562.0  
lgb4  0.923789  0.933892  0.923030          628.0  
lgb9  0.923561  0.938185  0.922788          402.0  
lgb3  0.922969  0.931672  0.922612          204.0  
lgb8  0.787390  0.789670  0.785937          145.0

## XGB

In [27]:
e.add_collector(
    ModelAttrCollector('xgb_feature_importance', Connector(processor=xgb.XGBClassifier), 'feature_importances', params={'importance_type': 'gain'})
)
e.add_collector(
    ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result')
)

# XGB with preprocessed stage features (ohe + std + ord)
e.set_node('xgb1', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.set_node('xgb2', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.set_node('xgb3', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc', 'max_depth': 4})
e.set_node('xgb4', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.075, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
# xgb5: ohe 제외 (categorical feature 없이)
e.set_node('xgb5', grp='xgb', edges={'X': [('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.exp()

Collect 1/1 (100%) Node 0
Collect 1/1 (100%) Node 0
Experimenting 5 node(s)
Exp 1/1 (100%) xgb5 5/5 (100%)> 1/10000 (0%) validation_0-auc: 0.7819, validation_1-auc: 0.78209229
Experimentation complete: 5 node(s)


In [28]:
d = e.collectors['xgb_evals_results'].get_attrs('xgb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('validation_1', 'auc')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
}, name=('evals_result', 'best_iteration'))

e.pipeline.compare_nodes(
    e.pipeline.get_node_names('xgb*')
)['XGBClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending=False)

params                 AUC                       evals_result
     learning_rate max_depth     valid train_sub valid_sub best_iteration
xgb3         0.100       4.0  0.923342  0.930310  0.922908         1111.0
xgb2         0.050   default  0.922984  0.933137  0.922393          810.0
xgb4         0.075   default  0.922855  0.930193  0.922106          392.0
xgb1         0.100   default  0.922743  0.933216  0.922059          393.0
xgb5         0.100   default  0.806164  0.826120  0.804998          387.0

## Catboost

In [29]:
e.add_collector(
    ModelAttrCollector('cb_feature_importance', Connector(processor=cb.CatBoostClassifier), 'feature_importances_pvc')
)
e.add_collector(
    ModelAttrCollector('cb_evals_results', Connector(processor=cb.CatBoostClassifier), 'evals_result')
)

# CB with raw features (native categorical handling)
e.set_node('cb1', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb2', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb3', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.1, 'depth': 4, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb4', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.075, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
# cb5: grade_subgrade 추가
e.set_node('cb5', grp='cb', edges={'X': [(None, X_num + X_cat + ['grade_subgrade'])]},
    params={'cat_features': X_cat + ['grade_subgrade'], 'iterations': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.exp()

Collect 1/1 (100%) Node 0
Collect 1/1 (100%) Node 0
Experimenting 5 node(s)
Exp 1/1 (100%) cb3 5/5 (100%)
Experimentation complete: 5 node(s)


In [30]:
d = e.collectors['cb_evals_results'].get_attrs('cb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('validation_1', 'AUC')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
}, name=('evals_result', 'best_iteration'))

e.pipeline.compare_nodes(
    e.pipeline.get_node_names('cb*')
)['CatBoostClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending=False)

params                         \
                                          cat_features    depth learning_rate   
cb3  [gender, marital_status, education_level, empl...      4.0         0.100   
cb4  [gender, marital_status, education_level, empl...  default         0.075   
cb1  [gender, marital_status, education_level, empl...  default         0.100   
cb2  [gender, marital_status, education_level, empl...  default         0.050   
cb5  [gender, marital_status, education_level, empl...  default         0.100   

                                                                                                                                                                       X  \
    DataSource [annual_income, credit_score, debt_to_income_ratio, education_level, employment_status, gender, interest_rate, loan_amount, loan_purpose, marital_status]   
cb3                                                 []                                                                                                                     
cb4                                                 []                                                                                                                     
cb1                                                 []                                                                                                                     
cb2                                                 []                                                                                                                     
cb5                                   [grade_subgrade]                                                                                                                     

          AUC                       evals_result  
        valid train_sub valid_sub best_iteration  
cb3  0.925199  0.927305  0.924605         2650.0  
cb4  0.925021  0.928971  0.924266         1921.0  
cb1  0.925007  0.930206  0.924551         1687.0  
cb2  0.924933  0.930010  0.924438         3335.0  
cb5  0.924929  0.929659  0.924333         1708.0

## Logistic Regression

In [31]:
from mllabs import col

In [32]:
for i, C in enumerate([1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3]):
    e.set_node(f'lr{i}', grp='lr', edges={'X': [('std', None), ('ohe', col.ohe_drop_first)]}, params={'C': C})
e.exp()

Experimenting 7 node(s)
Exp 1/1 (100%) lr3 7/7 (100%)
Experimentation complete: 7 node(s)


In [33]:
e.pipeline.compare_nodes(
    e.pipeline.get_node_names('lr*')
)['LogisticRegression'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
)

params       AUC                    
            C     valid train_sub valid_sub
lr0     0.001  0.910424  0.908343  0.909686
lr1     0.010  0.911870  0.909773  0.910948
lr2     0.100  0.912058  0.909939  0.911177
lr3     1.000  0.912075  0.909947  0.911209
lr4    10.000  0.912080  0.909947  0.911212
lr5   100.000  0.912081  0.909947  0.911213
lr6  1000.000  0.912081  0.909947  0.911213

In [34]:
from IPython.display import Markdown
Markdown(
    e.desc_node('lr1', show_params=True)
)

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr1["clf/lr/lr1"]
        lr1_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr><tr><td align='left'><b>C</b></td><td align='left'>0.01</td></tr></table>"]
    end
    style node_lr1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["pre/ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["pre/std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr></table>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    DataSource -->|y| node_lr1
    DataSource --> node_ohe
    DataSource --> node_std
    node_ohe --> node_lr1
    node_std --> node_lr1
```

**Path from DataSource to 'clf/lr/lr1' (3 path(s) found)**

### Edges

| Key | Node | Var |
|-----|------|-----|
| X | pre/std | * |
| X | pre/ohe | `<function ohe_drop_first at 0x7fd7fef6dee0>` |
| y | Data Source | `loan_paid_back` |

In [35]:
Markdown(
    e.desc_pipeline(max_depth = 2)
)

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_pre["pre"]
        node_ord["ord"]
        style node_ord fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lgb["lgb"]
            grp_lgb_count["9 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_xgb["xgb"]
            grp_xgb_count["5 node(s)"]
            style grp_xgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_xgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["5 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lr["lr"]
            grp_lr_count["7 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    DataSource --> grp_clf
    DataSource --> grp_pre
    grp_pre --> grp_clf
```

In [36]:
Markdown(
    e.desc_status()
)

**Experiment**: open

**Stage Nodes** (3)

| built |
| --- |
| 3 |

**Head Nodes** (26)

| built |
| --- |
| 26 |


## Stacking

In [9]:
stk_nodes = ['lgb5', 'lgb6', 'xgb3', 'cb3', 'cb4']
df_stk = e.collectors['stacking'].get_dataset(stk_nodes)
df_stk

,lgb5__loan_paid_back_1,lgb6__loan_paid_back_1,xgb3__loan_paid_back_1,cb3__loan_paid_back_1,cb4__loan_paid_back_1,loan_paid_back
id,,,,,,
273800,0.989731,0.987147,0.991524,0.989597,0.990399,1
80655,0.847761,0.842738,0.848688,0.848191,0.836669,1
128381,0.579269,0.579363,0.548441,0.567375,0.535936,1
390748,0.004857,0.006244,0.003855,0.007075,0.005612,0
135730,0.979413,0.978891,0.973123,0.977825,0.975378,1
...,...,...,...,...,...,...
195202,0.994476,0.994819,0.996792,0.993755,0.994123,1
1490,0.762383,0.722172,0.776158,0.788440,0.809491,1
74083,0.900801,0.895980,0.713825,0.745050,0.806778,1


In [13]:
stk_features = [i for i in df_stk.columns if i != y]
df_stk[stk_features].corr()

,lgb5__loan_paid_back_1,lgb6__loan_paid_back_1,xgb3__loan_paid_back_1,cb3__loan_paid_back_1,cb4__loan_paid_back_1
lgb5__loan_paid_back_1,1.000000,0.998875,0.993265,0.995474,0.995385
lgb6__loan_paid_back_1,0.998875,1.000000,0.993007,0.995141,0.995069
xgb3__loan_paid_back_1,0.993265,0.993007,1.000000,0.993699,0.993624
cb3__loan_paid_back_1,0.995474,0.995141,0.993699,1.000000,0.998982
cb4__loan_paid_back_1,0.995385,0.995069,0.993624,0.998982,1.000000


In [33]:
# import shutil
# shutil.rmtree('exp/stk1')
from sklearn.linear_model import LogisticRegression

In [34]:
if os.path.exists('exp/stk1'):
    e2 = Experimenter.load('exp/stk1', df_stk)
else:
    e2 = Experimenter.create(
        df_stk, 'exp/stk1',
        sp = StratifiedKFold(5, shuffle=True, random_state=456),
        splitter_params = {'y': y}
    )

e2.add_collector(
    MetricCollector(
        'AUC', Connector(edges = {'y': [(None, y)]}),
        '.*' + y + '_1', roc_auc_score, include_train = True
    )
)

e2.set_grp('meta', role = 'head', method = 'predict_proba', edges = {'y': [(None, y)]})
e2.set_grp('lr', parent = 'meta', processor = LogisticRegression)

for i, C in enumerate([1e-3, 1e-2, 1e-1, 1, 10]):
    e2.set_node(f'lr{i}', grp='lr', edges = {'X': [(None, stk_features)]}, params = {'C': C})
e2.exp()

Loaded: 0 node(s), 1 group(s), 5 fold(s)
Experimenting 5 node(s)
Exp 5/5 (100%)> lr1 5/5 (100%)
Experimentation complete: 5 node(s)


In [35]:
e2.pipeline.compare_nodes(
    e2.pipeline.get_node_names('lr*')
)['LogisticRegression'].fillna('default').join(
    e2.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).sort_values(('AUC', 'valid'), ascending = False)

params       AUC          
          C     valid train_sub
lr4  10.000  0.925249  0.925252
lr3   1.000  0.925236  0.925242
lr2   0.100  0.925186  0.925189
lr1   0.010  0.925144  0.925145
lr0   0.001  0.925138  0.925137

# Finalize All Experiments Objects

In [ ]:
e.close_exp()

In [37]:
e2.close_exp()

Finalize 'lr3'
Finalize 'lr0'
Finalize 'lr2'
Finalize 'lr4'
Finalize 'lr1'


# Train and Predict

In [38]:
e.add_trainer('trainer')

In [39]:
e.trainers['trainer'].select_head(stk_nodes)

In [40]:
e.trainers['trainer'].train()

Train 0
Train complete: 0 node(s)


In [41]:
next(e.trainers['trainer'].node_objs['cb3'].get_obj())

(<mllabs._node_processor.PredictProcessor at 0x7fd02760faa0>,
 None,
 {'build_id': '6ac1630e-5e29-4e44-864d-506b64cf4a14',
  'fit_time': 415.12092995643616,
  'train_shape': (534594, 10),
  'train_v_shape': (59400, 10)})

In [42]:
for i in e.trainers['trainer'].get_node_output('cb3'):
    print(i[0].data)

        cb3__loan_paid_back_0  cb3__loan_paid_back_1
id                                                  
156038               0.949942               0.050058
312444               0.995434               0.004566
238399               0.299298               0.700702
175164               0.092618               0.907382
471513               0.047280               0.952720
...                       ...                    ...
263857               0.150361               0.849639
468315               0.042276               0.957724
444940               0.013530               0.986470
239144               0.047015               0.952985
219282               0.026624               0.973376

[534594 rows x 2 columns]


In [43]:
for i in e.trainers['trainer'].process(df_test):
    print(i.data)

        cb3__loan_paid_back_0  cb3__loan_paid_back_1  lgb6__loan_paid_back_0  \
id                                                                             
593994               0.056486               0.943514                0.071552   
593995               0.026938               0.973062                0.020950   
593996               0.644989               0.355011                0.437707   
593997               0.089542               0.910458                0.058440   
593998               0.034213               0.965787                0.029150   
...                       ...                    ...                     ...   
848558               0.008054               0.991946                0.007554   
848559               0.168830               0.831170                0.127508   
848560               0.031477               0.968523                0.020507   
848561               0.017036               0.982964                0.018405   
848562               0.093283           

In [44]:
e.trainers['trainer'].node_objs['std'].objs_[0]

(<mllabs._node_processor.TransformProcessor at 0x7fd0c5956750>,
 None,
 {'build_id': '3a5b1d58-d80a-41a7-b267-03b382003fba',
  'fit_time': 0.01080632209777832,
  'train_shape': (534594, 5),
  'train_v_shape': (59400, 5)})

In [45]:
e2.add_trainer('trainer')

In [46]:
e2.trainers['trainer'].select_head(['lr4'])

In [47]:
e2.trainers['trainer'].train()

lr4 1/1 (100%) Split 1/1 (100%)
Train complete: 1 node(s)


In [51]:
next(
    e2.trainers['trainer'].process(
        next(e.trainers['trainer'].process(df_test, v=[slice(-1, None)])).data
    )
).data

,lr4__loan_paid_back_0,lr4__loan_paid_back_1
id,,
593994,0.055534,0.944466
593995,0.043024,0.956976
593996,0.708916,0.291084
593997,0.061352,0.938648
593998,0.045696,0.954304
...,...,...
848558,0.038475,0.961525
848559,0.103193,0.896807
848560,0.044196,0.955804


In [60]:
infr = e.trainers['trainer'].to_inferencer(v=[slice(-1, None)])
infr_meta = e2.trainers['trainer'].to_inferencer(v=[slice(-1, None)])

In [66]:
df_result = infr_meta.process(
    infr.process(df_test).data
).data
df_result.head()

,lr4__loan_paid_back_1
id,
593994,0.944466
593995,0.956976
593996,0.291084
593997,0.938648
593998,0.954304


In [71]:
df_result.rename(columns = lambda x: y).to_csv('data/submission.csv')

In [ ]:
!kaggle competitions submit -c playground-series-s5e11 -f data/submission.csv -m "Test"